 # Preprocessing 2: normalization & lemmatization

Pipeline:

1) Normalization
    - Lower case
    - Check unicode + removing unsupported characters
    - Sentence splitting
    - Handling "borken" words (e.g., words with "-" inside)
    - Double spaces normalization
    - Punctuation normalization 


2) Text preparation  for FastTextIt
    - Tokenization
    - Punctuation removal


In [ ]:
import re
import unicodedata
from pathlib import Path   
from typing import List
import spacy
import os

# carica il modello una sola volta (fuori dalla funzione)
nlp = spacy.load("it_core_news_sm", disable=["parser", "ner"])

1) Normalization & lemmatization

In [ ]:

def pulisci_testo(testo: str) -> str:
  

    # normalizzazione Unicode (accenti, caratteri strani)
    testo = unicodedata.normalize("NFKC", testo)

    # lowercase      
    testo = testo.lower()
    
    # normalizzazione apostrofi e virgolette
    testo = testo.replace("’", "'").replace("‘", "'")
    testo = testo.replace("“", '"').replace("”", '"')

    # rimozione parole spezzate da trattino a capo
    # tipo se abbiamo: "pa-\nrola" → "parola"
    testo = re.sub(r"-\s*\n\s*", "", testo)

    # rimozione caratteri non supportati
    testo = re.sub(r"[^a-zàèéìòù0-9.,;:!?'\-\s\"]", " ", testo)

    # normalizzazione spazi multipli
    testo = re.sub(r"\s+", " ", testo)

    # split in frasi (una per riga)
    frasi = re.split(r"(?<=[.!?])\s+", testo)

    # faccio qui lemmatizzazione
    frasi_lemmatizzate = []

    for frase in frasi:
        doc = nlp(frase)
        lemmi = [
            token.lemma_
            for token in doc
            if not token.is_space
        ]
        frasi_lemmatizzate.append(" ".join(lemmi))

    testo_finale = "\n".join(frasi_lemmatizzate)

    return testo_finale.strip()

2) Tokenization & Punctuation removal

In [3]:
def tokenizza_e_rimuovi_punteggiatura(testo: str) -> List[str]:
    # sostituiamo la punteggiatura con spazio (apostrofi OK)
    testo = re.sub(r"[.,;:!?\"()\[\]{}<>]", " ", testo)

    # normalizziamo di nuovo gli spazi
    testo = re.sub(r"\s+", " ", testo).strip()

    # tokenizziamo
    tokens = testo.split(" ")

    return tokens

In [4]:
def testo_per_fasttext(testo: str) -> str:
    """
    Restituisce il testo pronto per FastText:
    - tokenizza ogni frase
    - una frase per riga
    """
    frasi = testo.split("\n")
    righe_tokenizzate = []

    for frase in frasi:
        tokens = tokenizza_e_rimuovi_punteggiatura(frase)
        if tokens:
            righe_tokenizzate.append(" ".join(tokens))

    return "\n".join(righe_tokenizzate)


In [ ]:
for file in os.listdir("../year_clean/"):
    
    print(f"File: {file} ", end="")
    
    with open(f"../ocr_clean/{file}", "r", encoding="utf-8") as f:
        testo = f.read()

    testo_pulito = pulisci_testo(testo)
    testo_fasttext = testo_per_fasttext(testo_pulito)

    filename_without_extension = Path(file).stem
    
    with open(f"../lemmas/{filename_without_extension}_lemmas.txt", "w", encoding="utf-8") as f:
        f.write(testo_fasttext)

print("OK")

OK
